In [33]:
from datasets import load_dataset

raw_dataset = load_dataset("parquet", data_files={
    "train": "data/train-00000-of-00001.parquet",
    "validation": "data/validation-00000-of-00001.parquet",
})

In [35]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 90027
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 9936
    })
})

In [36]:
import numpy as np

def sample_by_title(dataset, max_per_title=100):
    rng = np.random.default_rng(42)  # 固定种子保证可复现
    indices = []
    # 按 title 分组
    title_to_indices = {}
    for i, title in enumerate(dataset["title"]):
        title_to_indices.setdefault(title, []).append(i)

    for title, idxs in title_to_indices.items():
        if len(idxs) <= max_per_title:
            indices.extend(idxs)
        else:
            indices.extend(rng.choice(idxs, max_per_title, replace=False).tolist())

    return dataset.select(sorted(indices))

raw_dataset["train"] = sample_by_title(raw_dataset["train"])
raw_dataset["validation"] = sample_by_title(raw_dataset["validation"],200)

In [37]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 43188
    })
    validation: Dataset({
        features: ['id', 'title', 'context', 'question', 'answers'],
        num_rows: 6859
    })
})

In [38]:
from transformers import AutoTokenizer
model_checkpoint = "bert-base-chinese"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [39]:
max_length = 384
stride = 128

def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping = inputs.pop("offset_mapping")
    sample_map = inputs.pop("overflow_to_sample_mapping")
    answers = examples["answers"]
    start_positions = []
    end_positions = []

    for i, offset in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = answers[sample_idx]
        # 空答案 → 不可回答
        if len(answer["answer_start"]) == 0:
            start_positions.append(0)
            end_positions.append(0)
            continue
        start_char = answer["answer_start"][0]
        end_char = answer["answer_start"][0] + len(answer["text"][0])
        sequence_ids = inputs.sequence_ids(i)

        # 找到上下文的起始和结束
        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        # 如果答案不完全在上下文内,标签为(0, 0)
        if offset[context_start][0] > start_char or offset[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            # 否则,它就是起始和结束 tokens 的位置
            idx = context_start
            while idx <= context_end and offset[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)

            idx = context_end
            while idx >= context_start and offset[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions
    return inputs

In [40]:
train_dataset = raw_dataset["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_dataset["train"].column_names,
)
len(raw_dataset["train"]), len(train_dataset)

Map:   0%|          | 0/43188 [00:00<?, ? examples/s]

(43188, 45998)

In [42]:
def preprocess_validation_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []
    answers = examples["answers"]  # 加这行

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])

        sequence_ids = inputs.sequence_ids(i)
        offset = inputs["offset_mapping"][i]
        inputs["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]
        
        # 空答案 → answer 字段为空
        answer = answers[sample_idx]
        if len(answer["answer_start"]) == 0:
            if "answer" not in inputs:
                inputs["answer"] = []
            inputs["answer"].append({"text": [], "answer_start": []})
        else:
            if "answer" not in inputs:
                inputs["answer"] = []
            inputs["answer"].append(answer)

    inputs["example_id"] = example_ids
    return inputs

In [43]:
validation_dataset = raw_dataset["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_dataset["validation"].column_names,
)
len(raw_dataset["validation"]), len(validation_dataset)

Map:   0%|          | 0/6859 [00:00<?, ? examples/s]

(6859, 7311)

In [44]:
from tqdm.auto import tqdm
import collections
import numpy as np
n_best = 20
max_answer_length = 30

import evaluate
metric = evaluate.load("squad_v2")
# 空答案判定阈值，CLS (0,0) 得分高于最佳答案得分+阈值则不答
null_answer_threshold = 0.0

def compute_metrics(start_logits, end_logits, features, examples):
    example_to_features = collections.defaultdict(list)
    for idx, feature in enumerate(features):
        example_to_features[feature["example_id"]].append(idx)

    predicted_answers = []
    for example in tqdm(examples):
        example_id = example["id"]
        context = example["context"]
        answers = []

        best_null_score = -float("inf")
        # 循环遍历与该示例相关联的所有特征
        for feature_index in example_to_features[example_id]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            null_score = start_logit[0] + end_logit[0]
            if null_score > best_null_score:
                best_null_score = null_score
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -n_best - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -n_best - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    # 跳过不完全位于上下文中的答案
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    # 跳过长度小于 0 或大于 max_answer_length 的答案
                    if (
                        end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    answer = {
                        "text": context[offsets[start_index][0] : offsets[end_index][1]],
                        "logit_score": start_logit[start_index] + end_logit[end_index],
                    }
                    answers.append(answer)



        if len(answers) > 0:
            best_answer = max(answers, key=lambda x: x["logit_score"])
            best_score = best_answer["logit_score"]
            # softmax 计算不答概率
             # 改用 best_null_score
            scores = np.array([best_null_score, best_score])
            scores -= scores.max()
            no_prob = float(np.exp(scores[0]) / (np.exp(scores[0]) + np.exp(scores[1])))
    
            if best_null_score > best_score + null_answer_threshold:
                predicted_answers.append({
                    "id": example_id,
                    "prediction_text": "",
                    "no_answer_probability": no_prob,
                })
            else:
                predicted_answers.append({
                    "id": example_id,
                    "prediction_text": best_answer["text"],
                    "no_answer_probability": no_prob,
                })
        else:
            predicted_answers.append({
                "id": example_id,
                "prediction_text": "",
                "no_answer_probability": 1.0,
            })


    theoretical_answers = [{"id": ex["id"], "answers": ex["answers"]} for ex in examples]
    return metric.compute(predictions=predicted_answers, references=theoretical_answers)

In [46]:
from transformers import TrainingArguments

args = TrainingArguments(
    "rag-qa-base-bert",
    eval_strategy="no",
    save_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    push_to_hub=False,
)

In [10]:
from transformers import Trainer
from transformers import AutoModelForQuestionAnswering

# 加载未训练的模型
raw_model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

trainer = Trainer(
    model=raw_model,
    args=TrainingArguments("tmp", per_device_eval_batch_size=16, fp16=True),
)

# 获取 logits
predictions = trainer.predict(validation_dataset)
start_logits, end_logits = predictions.predictions

# 调用你的 compute_metrics
metrics = compute_metrics(
    start_logits, end_logits,
    validation_dataset, raw_dataset["validation"],
)
metrics


Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/9936 [00:00<?, ?it/s]

{'exact': 0.020128824476650563,
 'f1': 0.03258952534314853,
 'total': 9936,
 'HasAns_exact': 0.02505637684790779,
 'HasAns_f1': 0.056078557707222206,
 'HasAns_total': 3991,
 'NoAns_exact': 0.01682085786375105,
 'NoAns_f1': 0.01682085786375105,
 'NoAns_total': 5945,
 'best_exact': 59.8329307568438,
 'best_exact_thresh': 0.0,
 'best_f1': 59.8329307568438,
 'best_f1_thresh': 0.0}

In [11]:
from transformers import pipeline
question_answerer = pipeline("question-answering", model=raw_model, tokenizer=tokenizer)
context = """
碧昂丝·吉赛尔·诺尔斯·卡特（生于1981年9月4日）是美国歌手、作曲家、唱片制作人和女演员。她在得克萨斯州休斯顿出生长大，小时候参加过各种歌舞比赛，上世纪90年代末以R&B女团“命运之子”的主唱而声名鹊起。由她父亲马修·诺尔斯（Mathew Knowles）管理的这个集团，一直以来都是世界上最畅销的女孩集团之一。暂停期间，碧昂丝发行了首张专辑《恋爱中的危险》（2003），确立了她作为全球独唱艺术家的地位，获得了五项格莱美奖，并在广告牌上热播100首单曲《疯狂恋爱》和《小男孩》。
"""
question = "碧昂丝在成长过程中，在哪些领域竞争？"
question_answerer(question=question, context=context)

Device set to use cuda:0


{'score': 8.68576971697621e-05, 'start': 148, 'end': 154, 'answer': '最畅销的女孩'}

In [47]:
from transformers import Trainer
from transformers import AutoModelForQuestionAnswering
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    tokenizer=tokenizer,
)
trainer.train()

Some weights of BertForQuestionAnswering were not initialized from the model checkpoint at bert-base-chinese and are newly initialized: ['qa_outputs.bias', 'qa_outputs.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\27729\AppData\Local\Temp\ipykernel_21688\1053338247.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
500,2.288300
1000,1.670700
1500,1.502500
2000,1.504600
2500,1.474600
3000,1.409300
3500,1.417400
4000,1.390000
4500,1.339700
5000,1.275700


TrainOutput(global_step=17250, training_loss=0.9973347707554914, metrics={'train_runtime': 5008.4636, 'train_samples_per_second': 27.552, 'train_steps_per_second': 3.444, 'total_flos': 2.704303848677069e+16, 'train_loss': 0.9973347707554914, 'epoch': 3.0})

In [48]:
# 获取 logits
predictions = trainer.predict(validation_dataset)
start_logits, end_logits = predictions.predictions

# 调用你的 compute_metrics
metrics = compute_metrics(
    start_logits, end_logits,
    validation_dataset, raw_dataset["validation"],
)
metrics

  0%|          | 0/6859 [00:00<?, ?it/s]

{'exact': 58.711182388103225,
 'f1': 58.7488457987073,
 'total': 6859,
 'HasAns_exact': 34.578402366863905,
 'HasAns_f1': 34.67393984220908,
 'HasAns_total': 2704,
 'NoAns_exact': 74.41636582430806,
 'NoAns_f1': 74.41636582430806,
 'NoAns_total': 4155,
 'best_exact': 63.58069689459105,
 'best_exact_thresh': 8.853434701450169e-05,
 'best_f1': 63.59284638188268,
 'best_f1_thresh': 8.853434701450169e-05}

In [73]:
from transformers import pipeline
question_answerer = pipeline("question-answering", model=model, tokenizer=tokenizer)

Device set to use cuda:0


In [ ]:
def qa_with_no_answer(qa_pipeline, question, context):
    inputs = qa_pipeline.tokenizer(question, context, return_tensors="pt")
    inputs = {k: v.to(qa_pipeline.device) for k, v in inputs.items()}
    outputs = qa_pipeline.model(**inputs)
    
    start_logits = outputs.start_logits[0].detach().cpu().numpy()
    end_logits = outputs.end_logits[0].detach().cpu().numpy()

    
    null_score = start_logits[0] + end_logits[0]
    
    # 找最佳非空答案
    best_score = -float("inf")
    best_start, best_end = 0, 0
    for s in range(len(start_logits)):
        for e in range(s, min(s + 30, len(end_logits))):  # max_answer_length=30
            score = start_logits[s] + end_logits[e]
            if score > best_score and s > 0 and e > 0:  # 排除 CLS
                best_score = score
                best_start, best_end = s, e
    
    if null_score > best_score:
        return {"answer": "", "score": null_score, "no_answer": True}
    else:
        return qa_pipeline(question=question, context=context)


context = """
小明是曹操的爸爸，小兰是李逵的妈妈。
"""
question = "太阳在哪"
qa_with_no_answer(question_answerer,question=question, context=context)


{'answer': '', 'score': np.float32(2.1020508), 'no_answer': True}

In [69]:
question1 = "李逵母亲是谁"
qa_with_no_answer(question_answerer,question=question1, context=context)

{'score': 0.4801517128944397, 'start': 10, 'end': 12, 'answer': '小兰'}